<a href="https://colab.research.google.com/github/shivansh2310/Quantitative-Portfolio-Management/blob/main/Alpha_Signals_%26_Linear_Regularization_(Chapters_3%E2%80%934).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### A. The Anatomy of an Alpha Signal
Isichenko broadly categorizes alpha signals into two types:

* Momentum (Trend): The tendency of winning stocks to keep winning. Usually measured over medium-to-long horizons (1 month to 12 months).

* Mean Reversion: The tendency of stocks that moved too far, too fast, to snap back. Usually measured over very short horizons (1 day to 1 week).

### B. The Curse of Dimensionality & Overfitting
If you have 500 stocks, and you calculate 100 different momentum and reversion signals for each, your matrix is massive. If you run a standard Ordinary Least Squares (OLS) regression on this, the math will memorize the noise in the historical data rather than learning the underlying signal. Your backtest will look like a multi-billion dollar strategy, and your live trading will immediately lose money.

### C. The Fix: Regularization (Ridge vs. Lasso)
To prevent overfitting, quants use penalized regressions. We add a penalty term to the model's loss function based on the size of the coefficients (weights).

* Ridge (L2 Penalty): Punishes the squared size of the coefficients. It shrinks correlated features together, preventing any single signal from dominating the portfolio.

* Lasso (L1 Penalty): Punishes the absolute size of the coefficients. This is magical for quants because it performs feature selection—it forces the weights of useless, noisy signals to exactly 0.0, removing them from the model entirely.

## Engineering Signals & The Ridge Model

In [13]:
import pandas as pd
import numpy as np
import yfinance as yf
import warnings
from sklearn.linear_model import Ridge
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')

In [2]:
universe = {
    'AAPL': 'Tech', 'MSFT': 'Tech', 'NVDA': 'Tech', 'AMD': 'Tech', 'ORCL': 'Tech',
    'JPM': 'Fin', 'BAC': 'Fin', 'GS': 'Fin', 'MS': 'Fin', 'C': 'Fin',
    'XOM': 'Energy', 'CVX': 'Energy', 'COP': 'Energy', 'EOG': 'Energy', 'SLB': 'Energy',
    'JNJ': 'Health', 'UNH': 'Health', 'PFE': 'Health', 'ABBV': 'Health', 'MRK': 'Health'
}
tickers = list(universe.keys())

print("Fetching historical data... (Takes ~10 seconds)")
# Fetch 1 year of daily close prices
prices = yf.download(tickers, period="1y")['Close']
prices = prices[tickers] # Ensure column order

Fetching historical data... (Takes ~10 seconds)


[*********************100%***********************]  20 of 20 completed


In [3]:
# We use .shift(-1) because today's features must predict tomorrow's return
raw_returns = prices.pct_change()
forward_returns = raw_returns.shift(-1)

In [4]:
# Convert to a "Long" format DataFrame (Standard for ML pipelines)
df = forward_returns.unstack().reset_index()
df.columns = ['Ticker', 'Date', 'Fwd_Return']
df = df.dropna()

In [5]:
# Map the sectors
df['Sector'] = df['Ticker'].map(universe)

In [6]:
def neutralize_and_rank(daily_data):
    # Market Neutralization (Subtract cross-sectional mean)
    daily_data['Market_Mean'] = daily_data['Fwd_Return'].mean()
    daily_data['Market_Neutral'] = daily_data['Fwd_Return'] - daily_data['Market_Mean']

    # Sector Neutralization (Subtract sector mean from the market-neutral returns)
    sector_means = daily_data.groupby('Sector')['Market_Neutral'].transform('mean')
    daily_data['Idiosyncratic_Return'] = daily_data['Market_Neutral'] - sector_means

    # Rank Normalization (Scale between 0 and 1 to suppress outliers)
    daily_data['ML_Target'] = daily_data['Idiosyncratic_Return'].rank(pct=True)

    return daily_data

print("Applying Cross-Sectional Neutralization and Rank Normalization...")
# Apply the function group-by-group for every single day in the dataset
ml_dataset = df.groupby('Date', group_keys=False).apply(neutralize_and_rank)

Applying Cross-Sectional Neutralization and Rank Normalization...


In [8]:
# We must sort by Ticker and Date to calculate rolling features correctly
ml_dataset = ml_dataset.sort_values(['Ticker', 'Date'])

In [11]:
# Short-Term Mean Reversion (5-Day Return)
# Hypothesis: High 5-day return means it's overbought and will revert (negative weight expected)
ml_dataset['F_Reversion_5d'] = ml_dataset.groupby('Ticker')['Fwd_Return'].transform(lambda x: x.shift(1).rolling(5).sum())

# Medium-Term Momentum (21-Day Return)
# Hypothesis: High 1-month return means it's trending (positive weight expected)
ml_dataset['F_Momentum_1m'] = ml_dataset.groupby('Ticker')['Fwd_Return'].transform(lambda x: x.shift(1).rolling(21).sum())

# Daily Volatility (21-Day standard deviation)
ml_dataset['F_Volatility'] = ml_dataset.groupby('Ticker')['Fwd_Return'].transform(lambda x: x.shift(1).rolling(21).std())

ml_dataset = ml_dataset.dropna()

print("Cross-Sectional Feature Standardization (Z-Scoring)...")
# Just like targets, features must be standardized cross-sectionally every single day
features = ['F_Reversion_5d', 'F_Momentum_1m', 'F_Volatility']

Cross-Sectional Feature Standardization (Z-Scoring)...


In [12]:
ml_dataset.head()

,Ticker,Date,Fwd_Return,Sector,Market_Mean,Market_Neutral,Idiosyncratic_Return,ML_Target,F_Reversion_5d,F_Momentum_1m,F_Volatility
21,AAPL,2025-07-16,-0.000666,Tech,0.005988,-0.006654,-0.011427,0.10,-0.004529,0.068741,0.010842
22,AAPL,2025-07-17,0.005523,Tech,-0.008428,0.013951,0.012663,0.90,-0.011210,0.058047,0.010760
23,AAPL,2025-07-18,0.006156,Tech,-0.003378,0.009533,0.007659,0.90,0.000198,0.077581,0.010059
24,AAPL,2025-07-21,0.009036,Tech,0.006188,0.002849,0.021547,1.00,0.018383,0.078932,0.010071
25,AAPL,2025-07-22,-0.001166,Tech,0.016324,-0.017491,-0.013924,0.05,0.025070,0.065483,0.009211


In [14]:
def standardize_features(daily_data):
    for f in features:
        daily_data[f] = (daily_data[f] - daily_data[f].mean()) / (daily_data[f].std() + 1e-8)
    return daily_data

ml_dataset = ml_dataset.groupby('Date', group_keys=False).apply(standardize_features)

print("Training the Penalized Linear Model (Ridge)...")
# X is our features, y is our Neutralized Rank Target from Day 1
X = ml_dataset[features]
y = ml_dataset['ML_Target']

# Initialize Ridge Regression with a heavy penalty (alpha)
# In statsmodels it's lambda; in sklearn it's alpha
ridge_model = Ridge(alpha=100.0)
ridge_model.fit(X, y)

Training the Penalized Linear Model (Ridge)...


Ridge(alpha=100.0)

In [17]:
# Print the learned coefficients
print("\nLEARNED ALPHA WEIGHTS (Ridge L2):")
print("="*45)
for feature, coef in zip(features, ridge_model.coef_):
    print(f"{feature:>15}: {coef:.6f}")
print("="*45)

print("\nEvaluating the Model (Information Coefficient - IC)...")
# Generate predictions
ml_dataset['Prediction'] = ridge_model.predict(X)

# Calculate daily IC (Spearman Rank Correlation between Prediction and True Target)
def calculate_daily_ic(daily_data):
    # Only calculate if we have enough variance
    if daily_data['Prediction'].std() < 1e-8:
        return np.nan
    ic, p_val = spearmanr(daily_data['Prediction'], daily_data['ML_Target'])
    return ic

daily_ic = ml_dataset.groupby('Date').apply(calculate_daily_ic).dropna()
mean_ic = daily_ic.mean()

print(f"Mean Daily Information Coefficient (IC): {mean_ic:.4f}")
if mean_ic > 0.02:
    print("STATUS: Excellent. You have a statistically significant edge.")
elif mean_ic > 0.0:
    print("STATUS: Marginal edge. Needs more features.")
else:
  print("STATUS: Model has inverted or no edge.")


LEARNED ALPHA WEIGHTS (Ridge L2):
 F_Reversion_5d: 0.000479
  F_Momentum_1m: -0.006323
   F_Volatility: 0.001590

Evaluating the Model (Information Coefficient - IC)...
Mean Daily Information Coefficient (IC): 0.0155
STATUS: Marginal edge. Needs more features.
